In [1]:
import sys
sys.path.append('..')
from config import DB_USER, DB_PASSWORD, DB_HOST, DB_PORT, DB_NAME
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine(
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

query = """
SELECT 
    r.grid_position,
    r.finish_position,
    r.points,
    d.driver_name,
    ra.season,
    ra.round,
    ra.circuit_name
FROM results r
JOIN drivers d ON r.driver_id = d.driver_id
JOIN races ra ON r.race_id = ra.race_id
"""

df = pd.read_sql(query, con=engine)
print(df.shape)
df.head()

(479, 7)


,grid_position,finish_position,points,driver_name,season,round,circuit_name
0,1,1,25.0,Lando Norris,2025,1,Australian Grand Prix
1,3,2,18.0,Lando Norris,2025,2,Chinese Grand Prix
2,2,2,18.0,Lando Norris,2025,3,Japanese Grand Prix
3,6,3,15.0,Lando Norris,2025,4,Bahrain Grand Prix
4,10,4,12.0,Lando Norris,2025,5,Saudi Arabian Grand Prix


In [2]:
X = df[['grid_position']]
y = df['finish_position']

print(X.shape, y.shape)

(479, 1) (479,)


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

Training rows: 383
Test rows: 96


In [5]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print(f"Slope (coefficient): {model.coef_[0]:.3f}")
print(f"Intercept: {model.intercept_:.3f}")

Slope (coefficient): 0.672
Intercept: 3.440


In [6]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae:.2f} positions")
print(f"R² score: {r2:.3f}")

Mean Absolute Error: 3.76 positions
R² score: 0.297


In [7]:
import numpy as np

baseline_pred = X_test['grid_position'].values

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_r2 = r2_score(y_test, baseline_pred)

print(f"Baseline (finish = grid):")
print(f"  MAE: {baseline_mae:.2f} positions")
print(f"  R²: {baseline_r2:.3f}")
print()
print(f"Your model:")
print(f"  MAE: {mae:.2f} positions")
print(f"  R²: {r2:.3f}")

Baseline (finish = grid):
  MAE: 3.86 positions
  R²: 0.127

Your model:
  MAE: 3.76 positions
  R²: 0.297


In [8]:
results_summary = pd.DataFrame({
    'model': ['Baseline (finish=grid)', 'Linear Regression'],
    'mae': [baseline_mae, mae],
    'r2': [baseline_r2, r2]
})

results_summary.to_csv('../data/model_results.csv', index=False)
results_summary

,model,mae,r2
0,Baseline (finish=grid),3.864583,0.126740
1,Linear Regression,3.756793,0.296582
